# Mini-Projet NLP : Transformers et Système RAG

Ce notebook présente une exploration pratique des modèles **Transformers** pour différentes tâches de traitement automatique du langage naturel, puis l’implémentation d’un système **RAG** simple permettant d’améliorer les réponses d’un modèle de langage à l’aide d’un corpus externe.

Le projet est divisé en deux grandes parties :

1. Exploration des Transformers et tâches NLP.
2. Systèmes RAG (Retrieval-Augmented Generation).

## Environnement et bibliothèques utilisées

Dans ce notebook, nous utiliserons principalement les bibliothèques suivantes :

* `transformers` : pour charger et utiliser des modèles pré-entraînés.
* `sentence-transformers` : pour générer les embeddings des documents et des requêtes.
* `faiss` : pour indexer les embeddings et effectuer une recherche de similarité.
* `torch` : pour l’exécution des modèles de deep learning.
* `pandas` et `numpy` : pour la manipulation des données.
* `PyPDF2` ou `pypdf` : pour lire le contenu des documents PDF si nécessaire.

Ces bibliothèques permettent de construire une chaîne complète allant du traitement du texte jusqu’à la génération de réponses augmentées par des documents externes.

## Partie 1 : Exploration des Transformers et tâches NLP

Les Transformers sont des architectures de deep learning très utilisées en NLP.  
Ils permettent de traiter du texte en capturant les relations entre les mots grâce au mécanisme d’attention.

Dans cette partie, nous allons utiliser des modèles pré-entraînés disponibles sur HuggingFace afin de réaliser plusieurs tâches NLP sans entraîner un modèle depuis zéro.

In [5]:
# Installation des bibliothèques nécessaires
!pip install transformers datasets accelerate

## 1-a Importation des bibliothèques

In [6]:
import torch
from transformers import AutoTokenizer, AutoModel, pipeline

device = 0 if torch.cuda.is_available() else -1

print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

GPU disponible : True
Nom du GPU : Tesla T4


## 1-b Exploration du tokenizer et du modèle Transformer

Avant d’utiliser les pipelines simplifiés, il est important de comprendre comment un modèle Transformer reçoit le texte.

Un modèle Transformer ne comprend pas directement les phrases sous forme de texte brut.

Le texte passe d’abord par un tokenizer :

Texte brut → Tokens → Identifiants numériques → Modèle Transformer

Le tokenizer découpe le texte en mots ou sous-mots, puis transforme ces éléments en nombres appelés `input_ids`.

In [7]:
from transformers import AutoTokenizer, AutoModel

# Modèle BERT adapté au français
model_name = "dbmdz/bert-base-french-europeana-cased"

# Chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Chargement du modèle
model = AutoModel.from_pretrained(model_name)

# Exemple de phrase
texte = "Les architectures Transformers ont révolutionné le traitement du langage naturel."

# Encodage du texte
inputs = tokenizer(texte, return_tensors="pt")

print("Texte original :")
print(texte)

print("\nTokens générés :")
print(tokenizer.tokenize(texte))

print("\nIdentifiants numériques des tokens :")
print(inputs["input_ids"])

# Passage dans le modèle
outputs = model(**inputs)

print("\nForme de la dernière couche cachée :")
print(outputs.last_hidden_state.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dbmdz/bert-base-french-europeana-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Texte original :
Les architectures Transformers ont révolutionné le traitement du langage naturel.

Tokens générés :
['Les', 'architecture', '##s', 'Trans', '##for', '##mers', 'ont', 'révolution', '##né', 'le', 'traitement', 'du', 'langage', 'naturel', '.']

Identifiants numériques des tokens :
tensor([[    2,   519, 21475,   212,  4627,  3599, 26552,   507,  3663,  1187,
           354,  5153,   378,  7373,  6713,    18,     3]])

Forme de la dernière couche cachée :
torch.Size([1, 17, 768])


## 1-b Utilisation des pipelines HuggingFace

HuggingFace propose une fonction appelée `pipeline`.

Cette fonction permet d’utiliser facilement des modèles pré-entraînés pour différentes tâches NLP, sans écrire manuellement toutes les étapes internes.

Dans cette partie, nous testons quatre tâches :

1. Analyse de sentiment
2. Classification de texte
3. Question Answering
4. Résumé automatique

## 1-b-1 Analyse de sentiment

L’analyse de sentiment consiste à déterminer l’opinion exprimée dans un texte.

Le résultat peut être positif, négatif ou parfois neutre selon le modèle utilisé.

Exemple :

Texte : "Ce projet est intéressant."  
Résultat attendu : sentiment positif

In [ ]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device=device
)

textes = [
    "Ce film est absolument magnifique !",
    "Je n'ai pas du tout aimé cette expérience.",
]

for texte in textes:
    result = sentiment_pipeline(texte)
    print(f"Texte   : {texte}")
    print(f"Sentiment label : {result[0]['label']} , score : {result[0]['score']:.4f}\n")


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Texte   : Ce film est absolument magnifique !
Sentiment: positive (0.9395)

Texte   : Je n'ai pas du tout aimé cette expérience.
Sentiment: negative (0.8531)



### Analyse du résultat

Le modèle retourne généralement deux informations :

- `label` : la classe prédite, par exemple positif ou négatif.
- `score` : le niveau de confiance du modèle.

Plus le score est proche de 1, plus le modèle est confiant dans sa prédiction.